# Session 3 — 외교 언어의 *의미*, 두 개의 눈으로 읽기
### 주제 1: 우크라이나 전쟁 · 의미·뉘앙스 분석 (사전 + Claude)

> **오늘 한 문장:** "지난주엔 *문법(구조)* 을 셌다. 오늘은 *의미(뉘앙스)* 를 읽는다.
> 규칙 기반은 'mutual' 이라는 단어를 **셀 순 있지만**, *왜* 그 단어를 골랐는지는 못 읽는다.
> 그래서 오늘은 **두 개의 눈** — 코드(사전)와 Claude(LLM) — 을 동시에 쓴다."

지난주(S2)에서 우리는 spaCy로 **구조**를 측정했다: 능동/수동(직설성), 동사 강도, 주어 패턴.
구조는 '문법'이다. 하지만 외교 성명문의 진짜 무기는 **단어 선택의 뉘앙스**다.
- 중국이 "all parties(모든 당사자)" 라 쓸 때, 그건 *가해자를 지우는* 장치다.
- "we are deeply concerned" 의 'concerned' 와 'condemn' 사이엔 외교적 거리가 있다.

오늘의 목표:

1. **Mutuality Index(상호성 지수)** 를 직접 만들고 소스별로 비교한다 — **이 프로젝트의 핵심 발견**.
2. **Hedging Density(완곡어 밀도)** 를 만들고 비교한다.
3. **Claude 의미 분석** 으로 framing/blame_target/stance/뉘앙스를 JSON으로 뽑는다.
4. **규칙 기반 vs Claude** 를 교차 비교한다 — hybrid 방법론의 심장.

> 💡 **오늘의 방법론 한 줄:** **"계산은 코드, 해석은 LLM, 판단은 인간."**
> 셀 수 있는 건 코드로(결정론적·재현 가능), 읽어야 하는 건 Claude로, 최종 책임은 사람이 진다.

> 💡 **운영 방식:** 셀을 위에서 아래로 하나씩 실행. `# TODO` 가 보이면 직접 채우고,
> 바로 아래 `# CHECK` 셀을 실행해 `✅` 가 떠야 다음으로.

## Step 0 — 환경 설정
프로젝트 폴더를 찾고, Claude SDK(anthropic)를 설치한다.

In [ ]:
!pip install anthropic -q

In [ ]:
# ── 프로젝트 환경 자동 설정 (Colab / 로컬 공용) ───────────────────────
# 이 셀은 모든 세션 노트북 맨 위에 동일하게 들어간다. 그냥 실행만 하면 된다.
import os, sys, json, glob

def find_project():
    """data/backup/corpus_clean.json 이 있는 program4 폴더를 찾는다."""
    cands = [".", "program4", "..", "../program4", "/content/program4",
             "/content/edu/program4", os.path.expanduser("~/program4")]
    for c in cands:
        if os.path.exists(os.path.join(c, "data", "backup", "corpus_clean.json")):
            return os.path.abspath(c)
    return None

PROJECT = find_project()
if PROJECT is None:
    print("⚠️  데이터를 찾지 못했습니다. 아래 둘 중 하나로 해결하세요:")
    print("  (A) Colab: 좌측 파일창에 program4 폴더를 통째로 업로드")
    print("  (B) Colab: !git clone <이 강의 repo 주소>  후 다시 실행")
else:
    sys.path.insert(0, PROJECT)
    os.chdir(PROJECT)
    print("✅ 프로젝트 경로:", PROJECT)


## Step 1 — 지난주 복습 + 오늘의 입력 불러오기

S1에서 저장한 `data/ukraine_working.json`(85건)을 다시 쓴다. **"지난주 산출물이 이번 주 입력."**

> **복습 한 문장:** S2 = *구조=문법* (능동/수동, 동사 강도). S3 = *의미=뉘앙스* (단어를 *왜* 골랐나).

In [ ]:
import pandas as pd, json, os
ukr = pd.read_json(os.path.join(PROJECT, "data", "ukraine_working.json"))
print("우크라이나 문서:", len(ukr), "건")
print("소스별:", ukr.groupby("source").size().to_dict())
ukr[["source","event_id","date","title"]].head(3)

In [ ]:
# CHECK Step1 — 입력 데이터가 제대로 올라왔는지
try:
    assert len(ukr) > 50 and set(ukr["source"].unique()) >= {"UN","KR","CN","FR"}
    print("✅ PASS — 85건 내외, 4개 소스 모두 존재. 분석 시작 OK")
except Exception as e:
    print("❌ FAIL —", e)

## Step 2 — 왜 규칙 기반과 LLM을 *둘 다* 쓰는가 (3분)

| | 규칙 기반 (사전/spaCy) | LLM (Claude) |
|---|---|---|
| 잘하는 것 | **세기**: 'mutual' 이 1000단어당 몇 번 | **읽기**: *왜* 그 단어를 썼는지, framing |
| 결과 | 결정론적·재현 가능·공짜 | 맥락 의존·돈/키 필요·해석적 |
| 약점 | 의도·뉘앙스 못 읽음 | 매번 답이 살짝 다름(검증 필요) |

> **핵심:** 둘은 경쟁이 아니라 **교차검증**이다. 사전이 "상호성 높음"이라 하고
> Claude도 "양면적 톤"이라 하면 → **신뢰**. 둘이 어긋나면 → **사람이 직접 본다.**
> 오늘 Step 2~3은 *코드의 눈*, Step 4는 *Claude의 눈*, Step 5에서 둘을 겹쳐 본다.

## Step 3 — Mutuality Index (상호성 지수) 🎯 **오늘의 핵심**

### 개념
외교에서 **양면적/상호적 표현**("both sides", "all parties", "mutual", "dialogue"...)은
*가해자와 피해자를 같은 평면에 놓아* 책임 소재를 흐리는 장치다.
침략 전쟁에서 "모든 당사자가 자제해야 한다"고 하면, 침략자와 피침략자를 동급으로 만든다.

### 측정 방법
양면적 표현 사전을 만들고, **1000단어당 등장 횟수**(밀도)로 잰다.
밀도로 재는 이유? 문서 길이가 제각각이라(프랑스는 짧고 UN은 길다) **횟수만 세면 불공평**하다.

In [ ]:
# 상호성 사전 — 양면적/상호적 표현 목록 (toolkit 과 동일)
MUTUALITY_TERMS = [
    "mutual", "both sides", "all sides", "all parties", "shared",
    "common interest", "common ground", "win-win", "dialogue",
    "cooperation", "consultation", "peaceful", "negotiation",
    "political settlement", "all relevant parties", "common security",
]
print("상호성 표현", len(MUTUALITY_TERMS), "개:", MUTUALITY_TERMS[:5], "...")

### TODO — `mutuality_index` 직접 구현하기

밀도 공식: **(표현 등장 총횟수 / 전체 단어수) × 1000**.
아래 빈칸 두 개를 채워라.

In [ ]:
# TODO: 빈칸 2개를 채워라
def mutuality_index(text):
    """양면적/상호적 표현 밀도 (1000단어당)."""
    low = text.lower()
    words = max(len(low.split()), 1)
    # 사전의 각 표현이 본문에 몇 번 나오는지 센다 (0번이면 버림)
    hits = {t: low.count(t) for t in MUTUALITY_TERMS if low.count(t)}
    total = sum(hits.values())               # 표현 등장 총횟수
    return {
        "mutuality_index": round(total / words * ____, 2),   # ← 빈칸1: 1000단어당이므로?
        "mutuality_hits": hits, "n_mutuality": total,
    }

# 빠른 손 테스트
demo = "Both sides should pursue dialogue and a peaceful political settlement through cooperation."
print(mutuality_index(____))   # ← 빈칸2: 위 demo 문장을 넣어라

In [ ]:
# CHECK Step3 — 정의가 맞는지 검증
try:
    r = mutuality_index("Both sides should pursue dialogue and a peaceful political settlement.")
    assert r["n_mutuality"] >= 3, "양면적 표현이 덜 잡혔다"
    assert r["mutuality_index"] > 0
    print("✅ PASS — mutuality_index 작동:", r)
except Exception as e:
    print("❌ FAIL —", e)

<details><summary>💡 힌트 / 정답</summary>

```python
return {
    "mutuality_index": round(total / words * 1000, 2),  # 빈칸1 = 1000
    ...
}
print(mutuality_index(demo))                            # 빈칸2 = demo
```
밀도 = 비율 × 1000. "1000단어당 몇 번"으로 환산하면 문서 길이가 달라도 공정하게 비교된다.
</details>

### 소스별 상호성 비교 — 이게 헤드라인이다

In [ ]:
from collections import defaultdict
mut = defaultdict(list)
for _, d in ukr.iterrows():
    mut[d["source"]].append(mutuality_index(d["text"])["mutuality_index"])

print(f"{'source':8}{'평균 상호성(1000단어당)':>22}{'문서수':>8}")
rows = []
for s in ["CN","KR","UN","FR"]:
    avg = sum(mut[s])/len(mut[s])
    rows.append((s, round(avg,2), len(mut[s])))
    print(f"{s:8}{avg:>22.2f}{len(mut[s]):>8}")
mut_avg = {s:a for s,a,_ in rows}

> **읽어라.** 대략 **CN ≈ 10.1, KR ≈ 7.2, UN ≈ 3.7, FR ≈ 1.1**.
> 중국이 압도적 1위다. **"중국 외교언어 = 양면성"** 이라는 통념이 *실제 데이터로 재현*된 것이다.
> 프랑스는 거의 안 쓴다 — 가해자를 직접 지목하는 직설적 언어를 택했다는 신호.
>
> ⚠️ 단, 이건 *상관*이지 *의도*가 아니다. **왜** 중국이 그렇게 쓰는지는 코드가 못 답한다 → Step 4에서 Claude에게 묻는다.

In [ ]:
# CHECK Step3-b — 핵심 발견(중국>한국>UN>프랑스) 재현 확인
try:
    assert mut_avg["CN"] > mut_avg["KR"] > mut_avg["UN"] > mut_avg["FR"], "순서가 안 맞는다"
    assert mut_avg["CN"] > 8, "중국 상호성이 예상보다 낮다"
    print("✅ PASS — 핵심 발견 재현: CN > KR > UN > FR")
    print("   중국이 프랑스의", round(mut_avg["CN"]/max(mut_avg["FR"],0.01),1), "배")
except Exception as e:
    print("❌ FAIL —", e)

## Step 4 — Hedging Density (완곡어 밀도)

### 개념
**완곡/유보 표현**("may", "appears to", "reportedly", "we believe"...)은 단정을 피하는 장치다.
"러시아가 공격했다"가 아니라 "공격이 있었던 것으로 보인다(appears to)"라고 쓰면 책임 추궁이 약해진다.
상호성과 짝을 이루는 또 다른 '거리두기' 신호다.

In [ ]:
# 완곡어 사전 (toolkit 과 동일). 앞뒤 공백은 단어 경계를 흉내내는 트릭.
HEDGES = [
    "may ", "might ", "could ", "appears to", "seems to", "possibly",
    "perhaps", "allegedly", "reportedly", "we believe", "it is hoped",
    "would ", "likely", "potentially", "to some extent", "in some cases",
    "it seems", "arguably", "presumably", "apparently",
]

def hedging_density(text):
    """완곡/유보 표현 밀도 (1000단어당)."""
    low = " " + text.lower() + " "   # 양끝 공백: "may " 같은 패턴을 문장 끝에서도 잡으려고
    words = max(len(low.split()), 1)
    hits = {h.strip(): low.count(h) for h in HEDGES if low.count(h)}
    total = sum(hits.values())
    return {
        "hedging_density": round(total / words * 1000, 2),
        "hedging_hits": hits, "n_hedges": total,
    }

print(hedging_density("It appears Russia may have struck, reportedly damaging the grid."))

### TODO — 소스별 완곡어 밀도 비교 표 만들기
Step 3의 상호성 비교 루프를 그대로 베껴서 `hedging_density` 로 바꾸면 된다.

In [ ]:
# TODO: 빈칸 1개 — hedging_density 를 호출하라
hed = defaultdict(list)
for _, d in ukr.iterrows():
    hed[d["source"]].append(____(d["text"])["hedging_density"])   # ← 빈칸: 어떤 함수?

print(f"{'source':8}{'평균 완곡어 밀도':>16}")
hed_avg = {}
for s in ["UN","CN","FR","KR"]:
    avg = sum(hed[s])/len(hed[s]); hed_avg[s] = round(avg,2)
    print(f"{s:8}{avg:>16.2f}")

In [ ]:
# CHECK Step4
try:
    assert hed_avg["KR"] == min(hed_avg.values()), "한국이 최저 완곡어가 아니다"
    assert hed_avg["UN"] >= 2.0 or hed_avg["CN"] >= 2.0, "UN/CN 완곡어가 예상보다 낮다"
    print("✅ PASS — 한국이 가장 단정적(완곡어 최저), UN/CN 이 가장 완곡:", hed_avg)
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: 빈칸은 hedging_density")

<details><summary>💡 힌트 / 정답</summary>

```python
hed[d["source"]].append(hedging_density(d["text"])["hedging_density"])
```
대략 **UN/CN ≈ 2.5, FR ≈ 2.0, KR ≈ 0.8~1.3**. 한국은 짧고 단정적인 규탄 성명을 내고,
UN은 신중한 다자 언어를 쓴다. 상호성(중국 1위)과 완곡어(UN/중국 상위)는 *다른* 거리두기 전략임에 주목.
</details>

## Step 4.5 — 부정·범위 처리(negation scope): 사전의 *오탐* 잡기 ⚠️

### 문제: 사전은 부정을 못 본다
앞의 `mutuality_index` 는 본문에 `mutual`, `cooperation` 이 **나왔는지**만 센다.
그런데 외교문에는 이런 문장이 흔하다:

> *"There is **no** mutual interest and **no** cooperation here."*
> ("여기엔 상호 이익도, 협력도 **없다**.")

이 문장은 상호성을 **부정**하는데, 단순 카운트는 `mutual`+`cooperation` = **+2** 로 잡는다.
**거짓 양성(false positive)** — 측정값이 실제 의미와 반대로 부풀려진다.

In [ ]:
# 오탐을 눈으로 확인: 같은 문장, 부정처리 ON vs OFF
from diplo_analysis import mutuality_index   # toolkit 의 '검증된' 버전(부정처리 내장)

neg_text = "There is no mutual interest and no cooperation here."

on  = mutuality_index(neg_text)                          # handle_negation=True (기본)
off = mutuality_index(neg_text, handle_negation=False)   # 옛날 방식(부정 무시)

print("부정처리 ON  →", on["n_mutuality"], "건", on["mutuality_hits"])
print("부정처리 OFF →", off["n_mutuality"], "건", off["mutuality_hits"])
print()
print("→ OFF는 2건으로 오탐, ON은 0건으로 올바르게 무시한다.")

### 어떻게 고치나 — '부정어 창(window)' 검사
매칭된 표현 **바로 앞 좁은 구간**(약 32자)에 부정어(`not`/`no`/`never`/`without`...)가
있으면 그 카운트를 **버린다**. 핵심 아이디어만 떼어 보면:

```python
NEGATORS = [" not ", " no ", "n't", " never ", " without ", ...]
pre = low[max(0, i-32):i]              # 매칭 위치 i 앞 32자
if not any(g in pre for g in NEGATORS):
    n += 1                              # 부정어가 없을 때만 1 센다
```

> **왜 중요한가 — 이 프로젝트의 '검증(validity)' 테마.**
> 측정값이 *말하는 것을 정말 재는가* = **측정 타당성**. 부정처리는 사전 기반 지표가
> 반대 의미 문장에 속지 않게 해, "중국 상호성 1위" 같은 발견의 **신뢰도**를 떠받친다.
> (규칙 기반은 결정론적이라 좋지만, *이런 함정*은 사람이 직접 메워야 한다.)

### TODO — 긍정문은 그대로, 부정문은 0
아래 빈칸을 채워 **부정처리가 긍정문은 건드리지 않는다**는 것까지 확인하라.

In [ ]:
# TODO: 빈칸 1개 — 부정문에 handle_negation 옵션을 꺼서 '옛날 방식' 결과를 만들어라
pos_text = "We seek mutual cooperation and dialogue."
neg_text = "There is no mutual interest and no cooperation here."

pos_n     = mutuality_index(pos_text)["n_mutuality"]                 # 긍정문(처리 ON)
neg_on    = mutuality_index(neg_text)["n_mutuality"]                # 부정문(처리 ON)
neg_off   = mutuality_index(neg_text, handle_negation=____)["n_mutuality"]  # ← 빈칸: 부정처리를 끄려면?

print(f"긍정문(ON)  n={pos_n}   ← 정상적으로 잡혀야 함")
print(f"부정문(ON)  n={neg_on}   ← 0이어야 함(오탐 제거)")
print(f"부정문(OFF) n={neg_off}   ← 2, 처리 안 하면 오탐")

In [ ]:
# CHECK Step4.5
try:
    assert pos_n >= 2, "긍정문이 과하게 깎였다(처리가 너무 공격적)"
    assert neg_on == 0, "부정문 오탐이 안 걸러졌다"
    assert neg_off == 2, "부정처리 OFF가 옛날 방식(2)을 재현 못 함"
    print("✅ PASS — 긍정문은 보존, 부정문 오탐은 제거. 측정 타당성 ↑")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: 부정처리를 끄려면 handle_negation=False")

<details><summary>💡 힌트 / 정답</summary>

```python
neg_off = mutuality_index(neg_text, handle_negation=False)["n_mutuality"]  # 빈칸 = False
```
`handle_negation=True`(기본)는 매칭 앞 32자에 부정어가 있으면 그 카운트를 버린다 → 부정문 0.
`False` 로 끄면 옛날 단순 카운트(2)로 돌아간다. **긍정문은 둘 다 그대로** — 처리가 멀쩡한 데이터를 망치지 않음을 확인하는 게 이 CHECK의 핵심.
</details>

## Step 5 — Claude 의미 분석: *왜* 그 단어를 썼나

여기서부터 **Claude의 눈**이다. 사전은 "all parties 가 3번 나왔다"까지만 안다.
*그게 가해자를 지우는 framing 인지*는 의미를 읽어야 안다. 그걸 Claude에게 시킨다.

### 우리가 Claude에게 뽑게 할 5개 키 (SEMANTIC_SCHEMA — toolkit 과 동일)
| 키 | 뜻 |
|---|---|
| `framing` | 이 성명이 사건을 어떻게 규정하나 (누가 가해자/피해자/중립) |
| `blame_target` | 가해 주체를 명시했나? `named` / `implied` / `none` |
| `stance_strength` | 입장 강도 1(완곡)~5(강경) 정수 |
| `mutuality_tone` | 양측 균형 강조 정도 1(일방적)~5(매우 양면적) 정수 |
| `subtle_hedging` | 규칙으론 못 잡는 미묘한 유보 표현 인용 |

> **프롬프트 엔지니어링 2원칙 (오늘 꼭 기억):**
> 1. **출력 구조를 강제한다** — "JSON 객체 하나만 출력하라" + 키 목록을 명시.
> 2. **파싱은 방어적으로** — LLM이 앞뒤로 말을 붙여도 정규식으로 `{...}` 만 떼낸다.

In [ ]:
# 분석 스키마와 프롬프트 (toolkit 의 claude_semantic 과 동일 구조)
SEMANTIC_SCHEMA = {
    "framing": "한 문장: 이 성명이 사건을 어떻게 규정하는가(누가 가해자/피해자/중립)",
    "blame_target": "가해 주체를 명시했는가? (named / implied / none)",
    "stance_strength": "1(완곡)~5(강경) 정수",
    "mutuality_tone": "양측 균형 강조 정도 1(일방적)~5(매우 양면적) 정수",
    "subtle_hedging": "규칙으로 못 잡는 미묘한 유보가 있으면 인용",
}
for k, v in SEMANTIC_SCHEMA.items():
    print(f"  {k:16s} {v}")

### API 키 준비 + DEMO_MODE
이 노트북은 **키가 없어도 끝까지 돌아가야 한다**(채점·오프라인용).
그래서 키가 있으면 *실제 Claude 호출*, 없으면 `DEMO_MODE` 로 **미리 손으로 쓴 예시**를 돌려준다.

> **Colab에서 키 넣는 법:** 좌측 🔑 아이콘 → `ANTHROPIC_API_KEY` 추가 → 노트북 접근 허용.
> 그러면 아래 코드가 `userdata.get(...)` 로 자동으로 읽는다.

In [ ]:
# 키 탐색: Colab userdata → 환경변수 순. 없으면 DEMO_MODE.
client = None
DEMO_MODE = True
key = None
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    key = os.environ.get("ANTHROPIC_API_KEY")

if key:
    try:
        from anthropic import Anthropic
        client = Anthropic(api_key=key)
        DEMO_MODE = False
        print("🔑 API 키 감지 — 실제 Claude 호출 모드")
    except Exception as e:
        print("⚠️ anthropic SDK 문제, DEMO_MODE 로 진행:", e)
else:
    print("ℹ️ API 키 없음 → DEMO_MODE (미리 준비한 예시로 진행). 노트북은 끝까지 돌아간다.")

# 모델 선택 가이드(주석):
#   claude-sonnet-4-6           ← 기본(균형). 본 수업 기본값.
#   claude-haiku-4-5-20251001   ← 싸고 빠름. 85건 전체 일괄 분석 같은 대량 작업에.
#   claude-opus-4-8             ← 가장 똑똑. 사전과 Claude가 어긋나는 '어려운 검증 케이스'에만.
MODEL = "claude-sonnet-4-6"

In [ ]:
import re

# DEMO_MODE 폴백: 키가 없을 때 돌려줄, 손으로 쓴 현실적 예시들(스키마 키 그대로).
# 소스 성격을 반영: 한국=강경·named, 중국=양면적·none, 프랑스=직설.
_DEMO_BANK = {
    "KR": {
        "framing": "러시아를 명백한 가해자로 규정하고 우크라이나의 주권·영토 보전을 옹호한다",
        "blame_target": "named", "stance_strength": 4, "mutuality_tone": 1,
        "subtle_hedging": "직접적 제재 언급은 피하고 '국제사회와 함께'라는 완충 표현 사용",
    },
    "CN": {
        "framing": "특정 가해자 없이 '모든 당사자'의 자제와 대화를 촉구하는 중립적 균형 틀",
        "blame_target": "none", "stance_strength": 2, "mutuality_tone": 5,
        "subtle_hedging": "'관련국들의 정당한 안보 우려' 같은 표현으로 러시아 입장을 암묵 수용",
    },
    "FR": {
        "framing": "러시아의 침략을 국제법 위반으로 직설적으로 비난한다",
        "blame_target": "named", "stance_strength": 5, "mutuality_tone": 1,
        "subtle_hedging": "거의 없음 — 단정적이고 직설적",
    },
    "UN": {
        "framing": "민간인 피해와 인도적 위기를 중심에 두고 적대행위 중단을 촉구",
        "blame_target": "implied", "stance_strength": 3, "mutuality_tone": 3,
        "subtle_hedging": "'우려를 표한다(express concern)'로 비난 수위를 외교적으로 조절",
    },
}

def claude_semantic(text, client, model=MODEL, source=None):
    """Claude로 framing/blame/강도/상호성 의미 분석.
       DEMO_MODE(키 없음)면 손으로 쓴 예시를 돌려줘 노트북이 끝까지 돈다."""
    if DEMO_MODE or client is None:
        ex = dict(_DEMO_BANK.get(source, _DEMO_BANK["UN"]))
        ex["_demo"] = True
        return ex
    # ── 실제 호출 경로 (키 있을 때만) ──
    schema_desc = "\n".join(f"- {k}: {v}" for k, v in SEMANTIC_SCHEMA.items())
    prompt = (
        "다음 외교 성명문을 분석해 JSON으로만 답하라. 키:\n" + schema_desc +
        "\n\n성명문:\n\"\"\"\n" + text[:6000] + "\n\"\"\"\n"
        "반드시 JSON 객체 하나만 출력."
    )
    msg = client.messages.create(
        model=model, max_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = msg.content[0].text
    m = re.search(r"\{.*\}", raw, re.S)        # 방어적 파싱: {...} 만 떼낸다
    return json.loads(m.group(0)) if m else {"_raw": raw}

print("claude_semantic 준비 완료. DEMO_MODE =", DEMO_MODE)

### 소스별 1건씩 의미 분석해 보기
키가 있으면 진짜 Claude가, 없으면 DEMO 예시가 답한다. 둘 다 **같은 키(스키마)** 로 나온다.

In [ ]:
# 소스별 대표 1건을 골라 Claude(또는 DEMO) 의미 분석
sem_results = {}
for s in ["KR","CN","FR","UN"]:
    sub = ukr[ukr["source"]==s]
    if len(sub)==0:
        continue
    d = sub.iloc[0]
    res = claude_semantic(d["text"], client, source=s)
    sem_results[s] = res
    tag = " (DEMO)" if res.get("_demo") else ""
    print(f"\n[{s}] {d['event_id']} {d['date']}{tag}")
    print(f"  framing       : {res.get('framing')}")
    print(f"  blame_target  : {res.get('blame_target')}")
    print(f"  stance_strength: {res.get('stance_strength')}")
    print(f"  mutuality_tone : {res.get('mutuality_tone')}")
    print(f"  subtle_hedging : {res.get('subtle_hedging')}")

In [ ]:
# CHECK Step5 — Claude(또는 DEMO) 출력이 스키마 키를 모두 갖췄는지
try:
    need = set(SEMANTIC_SCHEMA.keys())
    for s, r in sem_results.items():
        assert need <= set(r.keys()), f"{s}: 키 누락"
        assert isinstance(r["stance_strength"], int) and 1 <= r["stance_strength"] <= 5
    print("✅ PASS — 4개 소스 모두 스키마 키 완비, stance 1~5 정수.",
          "(DEMO_MODE)" if DEMO_MODE else "(LIVE)")
except Exception as e:
    print("❌ FAIL —", e)

## Step 6 — Hybrid 교차 비교: 코드의 눈 vs Claude의 눈 🔬

이제 핵심이다. **규칙 기반 mutuality_index** 와 **Claude의 mutuality_tone** 을 나란히 본다.
- 둘 다 "상호성 높음"이라 하면 → **신뢰**(서로 다른 방법이 같은 결론).
- 어긋나면 → 🚩 **사람이 직접 그 문서를 읽어야 한다**(LLM 환각? 사전 누락? 진짜 미묘한 케이스?).

이게 **"계산은 코드, 해석은 LLM, 판단은 인간"** 의 실제 모습이다.

### TODO — 교차표 채우기
각 소스의 규칙 기반 평균 상호성(`mut_avg`)과 Claude의 `mutuality_tone` 을 나란히 둔다.
빈칸 하나만 채워라.

In [ ]:
# TODO: 빈칸 1개 — Claude 결과(sem_results)에서 mutuality_tone 을 꺼내라
print(f"{'source':8}{'규칙 mutuality':>16}{'Claude tone(1-5)':>18}  판정")
for s in ["CN","KR","UN","FR"]:
    rule = mut_avg[s]                       # 1000단어당 밀도
    tone = sem_results[s]["____"]           # ← 빈칸: Claude가 매긴 양면성 톤
    # 거칠게 정규화: 규칙 밀도 5 이상이면 '높음'으로 본다
    rule_hi = rule >= 5
    tone_hi = tone >= 4
    verdict = "✅ 일치" if rule_hi == tone_hi else "🚩 불일치 → 사람이 확인"
    print(f"{s:8}{rule:>16.2f}{tone:>18}  {verdict}")

In [ ]:
# CHECK Step6 — 교차 비교가 의미 있게 돌아갔는지
try:
    cn_tone = sem_results["CN"]["mutuality_tone"]
    fr_tone = sem_results["FR"]["mutuality_tone"]
    # 규칙(중국 최고)과 Claude(중국 톤 높음)가 같은 방향인지
    assert cn_tone >= fr_tone, "중국 톤이 프랑스보다 낮게 나왔다(점검 필요)"
    print("✅ PASS — 규칙(CN 상호성 1위)과 Claude(CN tone 최고)가 같은 방향. 교차검증 성립.")
    print("   → 서로 다른 두 방법이 같은 결론 = 발견의 신뢰도가 올라간다.")
except Exception as e:
    print("❌ FAIL —", e)

<details><summary>💡 힌트 / 정답</summary>

```python
tone = sem_results[s]["mutuality_tone"]
```
규칙 기반은 *밀도(숫자)*, Claude는 *톤(1~5 판단)*. 둘은 단위가 다르지만 **같은 현상**을 본다.
중국이 양쪽 모두에서 최고로 나오면, "중국 외교언어=양면성" 발견은 한 방법의 우연이 아니다.
어긋나는 칸이 있으면 그게 바로 **사람이 읽어야 할 문서** — hybrid의 가치는 거기서 나온다.
</details>

## Step 7 — Targeted Sentiment (대상별 태도): *누구를* 비판하나 🎯 새 차원

### 동기 — 상호성과 무엇이 다른가
- **Mutuality(상호성)** 는 *양면적 단어*를 센다 — "both sides", "mutual" 이 몇 번 나오나.
  → 텍스트의 **표면**(어휘)을 본다.
- **Targeted Sentiment(대상별 태도)** 는 한 발 더 들어가 묻는다:
  > **"한 성명 안에서 러시아 vs 우크라이나(또는 이스라엘 vs 팔레스타인)에 대한 감정이 다른가?"**
  행위자별로 문장을 모아 **부정어/긍정어**를 세고, 가장 부정적으로 다룬 **most_criticized 행위자**를 뽑는다.

→ 같은 '균형/편향'을 **독립된 두 번째 차원**에서 재는 것. Step 3의 발견을 *다른 방법*으로 재확인할 기회다.

In [ ]:
from diplo_analysis import targeted_sentiment

# 한 문장으로 감 잡기: 러시아엔 부정, 우크라이나엔 긍정
demo = ("Russia attacked civilians and we condemn this brutal aggression. "
        "We firmly support Ukraine's sovereignty and territorial integrity.")
print(targeted_sentiment(demo, "ukraine"))
# → Russia ≈ -1.0, Ukraine ≈ +1.0, most_criticized = "Russia"

### 두 주제(우크라이나+가자)로 넓혀 비교한다
대상별 태도의 진가는 **두 분쟁을 한 소스가 어떻게 다르게 비판하나**에서 나온다.
그래서 여기선 우크라이나뿐 아니라 **가자(gaza)** 성명도 백업 코퍼스에서 같이 불러온다.

In [ ]:
import json, os
from collections import defaultdict

corpus = json.load(open(os.path.join(PROJECT, "data", "backup", "corpus_clean.json"),
                       encoding="utf-8"))
two = [d for d in corpus if d["topic"] in ("ukraine", "gaza")]
print("우크라이나+가자 문서:", len(two), "건")

# 소스별로 'most_criticized 행위자' 분포를 센다
crit = defaultdict(lambda: defaultdict(int))
for d in two:
    r = targeted_sentiment(d["text"], d["topic"], d.get("lang", "en"))
    mc = r["most_criticized"]
    if mc:
        crit[d["source"]][mc] += 1

for s in ["UN", "FR", "CN", "KR"]:
    print(f"  {s}: {dict(crit[s])}")

### TODO — '양측 균등 비판' 점수 만들기
각 소스가 **가해자 쪽(Russia·Israel)** 과 **피해자/상대 쪽(Ukraine·Palestinians)** 중
어느 쪽을 더 비판했는지 본다. 균형 잡힌 소스일수록 양쪽 수가 비슷하다.
빈칸을 채워라.

In [ ]:
# TODO: 빈칸 1개 — '상대 쪽'(피침략/피공격 측) 비판 건수를 더하라
AGGRESSOR = {"Russia", "Israel"}        # 통상적 가해자 측
OTHER     = {"Ukraine", "Palestinians"} # 상대 측

print(f"{'source':8}{'가해자측 비판':>12}{'상대측 비판':>12}{'균형도':>10}")
for s in ["UN", "FR", "CN", "KR"]:
    agg = sum(crit[s][a] for a in AGGRESSOR)
    oth = sum(crit[s][a] for a in ____)        # ← 빈칸: 어느 집합을 합산?
    bal = round(min(agg, oth) / max(agg, oth, 1), 2)   # 1에 가까울수록 양측 균등
    print(f"{s:8}{agg:>12}{oth:>12}{bal:>10}")

In [ ]:
# CHECK Step7
try:
    cn_agg = sum(crit["CN"][a] for a in {"Russia","Israel"})
    cn_oth = sum(crit["CN"][a] for a in {"Ukraine","Palestinians"})
    un_oth = sum(crit["UN"][a] for a in {"Ukraine","Palestinians"})
    un_agg = sum(crit["UN"][a] for a in {"Russia","Israel"})
    cn_bal = min(cn_agg, cn_oth) / max(cn_agg, cn_oth, 1)
    un_bal = min(un_agg, un_oth) / max(un_agg, un_oth, 1)
    # 중국이 UN보다 양측을 더 균등하게 비판한다
    assert cn_bal > un_bal, "중국이 UN보다 균형적이지 않게 나왔다(점검)"
    print(f"✅ PASS — 중국 균형도 {cn_bal:.2f} > UN 균형도 {un_bal:.2f}")
    print("   → '중국만 양측을 거의 균등 비판'이 대상별 태도 차원에서도 재현됨")
except Exception as e:
    print("❌ FAIL —", e, "\n힌트: 빈칸은 OTHER")

<details><summary>💡 힌트 / 정답</summary>

```python
oth = sum(crit[s][a] for a in OTHER)   # 빈칸 = OTHER
```

**읽어라 (핵심 해석).** most_criticized 분포는 대략:
- **UN** {Russia 18, Israel 10} — 비판이 **가해자 측(러시아·이스라엘)에 집중**.
- **France** {Russia 18, Israel 12, Palestinians 13} — 역시 가해자 측에 무게(+팔레스타인 일부).
- **China** {Russia 11, Ukraine 6, Israel 8, Palestinians 9} — **양측을 거의 균등하게 비판**.
- **Korea** 작고 균형적(표본 적음).

→ UN·프랑스는 비판이 한쪽으로 쏠리는데 **중국만 양측을 고르게 비판**한다.
이건 Step 3에서 본 **mutuality(중국 최고)** 를, *완전히 다른 방법*(대상별 감정)으로 **독립 재확인**한 것이다.
한 차원의 우연이 아니라 **여러 차원에서 일관된 '균형 프레이밍'** = 발견의 신뢰도가 또 올라간다.
</details>

> 💡 **하이브리드 한 발 더 (선택).** 대상별 태도는 사전 기반이라 빠르고 결정론적이지만
> 비꼼·조건문은 못 읽는다. Claude에게 **aspect-based sentiment**("행위자별로 톤을 1~5로")를
> 시켜 교차검증할 수 있다 — 단, 이 노트북은 `DEMO_MODE` 면 호출하지 않고 넘어간다(키·비용 보호).
>
> ```python
> if not DEMO_MODE:
>     # client.messages.create(... "행위자별 감정을 JSON으로" ...)  # 실제 호출은 키 있을 때만
>     pass
> ```

## 🎯 회고 (5분)

1. **Mutuality Index** 에서 중국이 1위, 프랑스가 꼴찌였다. 두 나라의 *외교적 위치*가 이 차이를 어떻게 설명하나?
2. **밀도(1000단어당)** 로 잰 이유는? 만약 그냥 *횟수*로 셌다면 어느 소스가 부당하게 유리/불리했을까?
3. 규칙 기반과 Claude가 **어긋난 칸**이 있었다면, 너라면 다음에 무엇을 하겠나? (재현성과 책임의 관점에서)
4. **"계산은 코드, 해석은 LLM, 판단은 인간"** — 이 문장에서 '인간'이 빠지면 무슨 일이 생기나?

## ▶️ 다음 (Session 4)
> "오늘까지 우크라이나의 **6개 차원**(직설성·동사강도·주어 / 상호성·완곡어·침묵)이 다 모였다.
> 다음 주엔 이걸 **검증**(원문 URL 클릭, 표본 수작업 확인)하고 **시각화**한다.
> 레이더 차트 위에 4개 소스의 '외교 언어 지문(fingerprint)'을 겹쳐 그린다 — 우크라이나 1주제 완성.